In [1]:
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

def load_song_data(filepath='ClassicHit.csv'):

    if not os.path.exists(filepath):
        print(f"Error: The file '{filepath}' was not found.")
        return None, None, None
        
    try:
        df = pd.read_csv(filepath)
        df.dropna(subset=['Genre', 'Popularity'], inplace=True)
        df['Popularity'] = df['Popularity'].astype(int)
        
        numerical_features = df.select_dtypes(include='number').drop(columns=['Year', 'Popularity'], errors='ignore')
        
        # Normalise the values
        scaler = StandardScaler()
        scaled_features = scaler.fit_transform(numerical_features)
        
        scaled_df = pd.DataFrame(scaled_features, index=df.index, columns=numerical_features.columns)
        
        unique_genres = sorted(df['Genre'].unique())
        
        return df, scaled_df, unique_genres
    except Exception as e:
        print(f"An error occurred while loading or processing the data: {e}")
        return None, None, None

def get_user_genres(user_number, available_genres):

    print(f"\n--- User {user_number}, please make your selection ---")
    print("Available genres:", ", ".join(available_genres))
    
    selected_genres = []
    genre_map = {genre.lower(): genre for genre in available_genres}

    while len(selected_genres) < 3:
        prompt = f"Enter genre {len(selected_genres) + 1} of 3: "
        choice = input(prompt).strip().lower()
        
        if choice in genre_map:
            original_case_genre = genre_map[choice]
            if original_case_genre not in selected_genres:
                selected_genres.append(original_case_genre)
                print(f"Added '{original_case_genre}'.")
            else:
                print("You've already selected that genre. Please pick another.")
        else:
            print("Invalid genre. Please choose from the list of available genres.")
            
    return selected_genres

def recommend_songs_ml(original_df, scaled_df, user1_genres, user2_genres):
    
    print("\nCalculating your combined musical taste profile...")
    all_genres = list(set(user1_genres + user2_genres))
    
    # Get the songs from the selected genres
    genre_songs_indices = original_df[original_df['Genre'].isin(all_genres)].index
    
    if len(genre_songs_indices) == 0:
        return pd.DataFrame() 

    # Get the normalised features for these songs
    genre_features = scaled_df.loc[genre_songs_indices]
    
    # Create the "average taste profile" by calculating the mean of the features
    taste_profile = genre_features.mean(axis=0).values.reshape(1, -1)
    
    print("Finding the best song matches in the entire library...")
    # Calculate the cosine similarity between the taste profile and ALL songs in the library
    similarity_scores = cosine_similarity(taste_profile, scaled_df.values)
    
    # Get the indices of the top 10 most similar songs
    top_10_indices = similarity_scores[0].argsort()[-10:][::-1]
    
    # Get the recommended songs from the original dataframe
    recommendations = original_df.iloc[top_10_indices]
    
    print("\n--- Here are your AI-powered recommendations! ---")
    print("These songs were selected based on their musical similarity to your combined tastes.")
    return recommendations[['Track', 'Artist', 'Genre', 'Popularity']]


def main():
    original_df, scaled_df, unique_genres = load_song_data()
    
    if original_df is not None:
        user1_choices = get_user_genres(1, unique_genres)
        user2_choices = get_user_genres(2, unique_genres)
        
        recommended_tracks = recommend_songs_ml(original_df, scaled_df, user1_choices, user2_choices)
        
        if not recommended_tracks.empty:
            print(recommended_tracks.to_string(index=False))
        else:
            print("\nCould not find any songs matching the selected genres.")

if __name__ == "__main__":
    main()





--- User 1, please make your selection ---
Available genres: Alt. Rock, Blues, Country, Disco, EDM, Folk, Funk, Gospel, Jazz, Metal, Pop, Punk, R&B, Rap, Reggae, Rock, SKA, Today, World


Enter genre 1 of 3:  rock


Added 'Rock'.


Enter genre 2 of 3:  disco


Added 'Disco'.


...

Added 'Today'.


Enter genre 2 of 3:  edm


Added 'EDM'.


Enter genre 3 of 3:  funk


Added 'Funk'.

Calculating your combined musical taste profile...
Finding the best song matches in the entire library...

--- Here are your AI-powered recommendations! ---
These songs were selected based on their musical similarity to your combined tastes.
                                   Track                             Artist Genre  Popularity
Love Song - Givin' U Some Pleasure Remix                          Ofra Haza World           1
   Let It All Blow - Special Disco Remix                          Dazz Band  Funk          18
               Love Magic - 12" Club Mix John Davis & The Monster Orchestra Disco          18
                Guess You're Not The One                        George Duke  Funk          18
                        Three Card Molly                        Elvin Jones  Jazz           7
                 To the End of the World                  Pat Metheny Group World          33
                              No Mystery                     Stanley Clarke  Jazz    